# CS60075 Project 10: Open-domain QA

Runs the TriviaQA experiment end to end from the lightweight GitHub source ZIP.
Download the repository with **Code > Download ZIP**, then upload that small ZIP
in the first picker. CPU is supported; datasets, models, and Wikipedia require
internet and are downloaded at runtime. No video is needed.

**Default:** pinned CPU environment, 300 unique validation questions, seed 42,
fresh Wikipedia collection. This is a new experiment, so reference scores may
change. A separately retained snapshot can optionally restore the reference
corpus. All stages run as fresh Python subprocesses to avoid stale imports.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, subprocess, sys, os, json

base = Path.cwd().resolve()
uploaded = files.upload()  # Select the small GitHub source ZIP.
for name in uploaded:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            for entry in z.infolist():
                if not (base / entry.filename).resolve().is_relative_to(base):
                    raise ValueError("Unsafe ZIP entry")
            z.extractall(base)
# Accept both GitHub's repository-branch/ layout and the original package.
candidates = [p for p in [base, *base.iterdir()]
              if p.is_dir() and (p / "config.json").is_file()
              and (p / "code/run.py").is_file()]
assert len(candidates) == 1, "Upload one project source ZIP into a fresh runtime."
project = candidates[0]
os.chdir(project)
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Project:", project)


## Install the pinned CPU environment

The reference run uses CPU. Keep this default to reproduce it. You may use an optional GPU wheel below, but its outputs are a separate device run. Installation may take several minutes.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
# OPTIONAL GPU: run the following only if a CUDA runtime is selected.
# subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124", "--force-reinstall", "--no-deps"], check=True)
# The CUDA run was not validated as part of the reference CPU experiment.
subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], check=True)

## Optional: restore a separately retained corpus snapshot

The default `RESTORE_SNAPSHOT=False` downloads data and collects live Wikipedia
pages. Large data and the reference snapshot are not hosted on GitHub. To use
an existing `NLP_QA_Data_Snapshot.zip` from your separate project files, set the
flag to `True`. A live rebuild is a new experiment and may change the scores.


In [ ]:
RESTORE_SNAPSHOT = False
if RESTORE_SNAPSHOT and not (project / "data/corpus.jsonl").exists():
    snapshots = files.upload()  # Optional local reference snapshot only.
    for name in snapshots:
        if name.endswith(".zip"):
            with zipfile.ZipFile(name) as z:
                for entry in z.infolist():
                    relative = Path(entry.filename)
                    if relative.parts and relative.parts[0] == "nlp_qa_project":
                        relative = Path(*relative.parts[1:])
                    target = (project / relative).resolve()
                    if not target.is_relative_to(project):
                        raise ValueError("Unsafe ZIP entry")
                    if entry.is_dir():
                        target.mkdir(parents=True, exist_ok=True)
                    else:
                        target.parent.mkdir(parents=True, exist_ok=True)
                        target.write_bytes(z.read(entry))
    assert (project / "data/corpus.jsonl").is_file(), "Snapshot lacks data/corpus.jsonl"

# A live collection has new evidence. Preserve reference audit labels separately.
if not RESTORE_SNAPSHOT:
    labels = project / "results/support_labels.csv"
    if labels.exists():
        from datetime import datetime
        archived = labels.with_name("reference_support_labels_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".csv")
        labels.rename(archived)
        print("Reference support labels archived. Review the new support audit before reporting its accuracy.")

def run(stage):
    subprocess.run([sys.executable, "-u", "code/run.py", stage], check=True)

run("prepare")


## Task 1 - BM25 and extraction

Runs independently after preparation. It retrieves 50 candidates for evaluation, uses the top passage for answering, and saves every output.

In [ ]:
run("task1")

## Task 2 - Dense retrieval and RAG

Frozen MiniLM, normalized FAISS flat search, top 5 evidence passages, 200 model tokens per passage, greedy generation, max 64 new tokens.

In [ ]:
run("task2")

## Task 3 - Hybrid retrieval and support warnings

RRF constant 60, candidate depth 50, same answer generator. Analysis also applies the overlap check to original Task 2 outputs.

In [ ]:
run("task3")

## Tables, plots, qualitative examples, and PDF report

After changing the corpus, model, prompt, or predictions, independently re-review support labels. Remove stale support_labels.csv for a changed experiment; do not reuse old support judgments.

In [ ]:
run("analyze")
run("report")
from IPython.display import display, Markdown, Image
display(Markdown(Path("results/comparison.md").read_text()))
display(Image(filename="results/comparison.png"))

## Download the new run

This creates a local course-submission archive containing the newly generated
results. Keep it outside GitHub: prediction dumps can exceed the repository's
small-file limit. Large datasets and model checkpoints remain excluded from
the archive. The separate snapshot is needed for exact corpus restoration.


In [ ]:
subprocess.run([sys.executable, "code/package_submission.py"], check=True)
files.download("../NLP_QA_Submission.zip")
files.download("report.pdf")